# 01 · Data Ingestion, Quality Checks & Preprocessing
**Project:** Distributed App Store Analytics with Apache Spark  
**Goal:** load the raw Apple App Store CSV with an explicit schema, audit data quality and write a clean, feature-enriched Parquet table used by every later notebook.

| Step | What happens |
|---|---|
| 1 | Spark session with Adaptive Query Execution (AQE) |
| 2 | Column mapping + explicit schema enforcement |
| 3 | Data-quality audit: missing values, IQR outliers, business rules, duplicates |
| 4 | Cleaning + derived features (`is_free`, price / rating bands, `popularity_score`) |
| 5 | Write `data/processed/apps_processed.parquet` |

## 1. Spark session

In [ ]:
# ── Environment & Spark session (shared by every notebook) ─────────────────────
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/app-store-analytics-spark"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# Raw CSV location (override with the APPSTORE_CSV environment variable)
DATA_PATH = os.environ.get("APPSTORE_CSV", os.path.join(PROJECT_DIR, "data", "raw", "apple_appstore_apps.csv"))
PROCESSED_PATH = os.path.join(PROJECT_DIR, "data", "processed", "apps_processed.parquet")
FIG_DIR = os.path.join(PROJECT_DIR, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
         .appName("AppStoreAnalytics")
         .config("spark.sql.adaptive.enabled", "true")                     # Adaptive Query Execution
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge small shuffle partitions
         .config("spark.sql.adaptive.skewJoin.enabled", "true")            # split skewed partitions
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | Spark UI: {spark.sparkContext.uiWebUrl}")

## 2. Ingestion with an explicit schema
Public App Store exports use different column names, so the raw header is first mapped to a fixed **canonical schema**; values are then cast to explicit types. Missing columns fail loudly instead of silently falling back to demo data.

In [ ]:
# Canonical schema enforced on every downstream step
CANONICAL_SCHEMA = StructType([
    StructField("app_id", StringType(), False),
    StructField("app_name", StringType(), True),
    StructField("primary_genre", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_rating", DoubleType(), True),
    StructField("rating_count", LongType(), True),
    StructField("size_mb", DoubleType(), True),
    StructField("developer", StringType(), True),
])

# Accepted source names for each canonical column (case-insensitive)
COLUMN_CANDIDATES = {
    "app_id":        ["app_id", "appid", "id", "track_id"],
    "app_name":      ["app_name", "name", "track_name", "title"],
    "primary_genre": ["primary_genre", "prime_genre", "genre", "category"],
    "price":         ["price"],
    "user_rating":   ["user_rating", "average_user_rating", "averageuserrating", "rating"],
    "rating_count":  ["rating_count", "reviews", "rating_count_tot", "user_rating_count"],
    "size_mb":       ["size_mb", "size_bytes"],
    "developer":     ["developer", "seller_name", "developer_name"],
}

if not os.path.exists(DATA_PATH) and not DATA_PATH.startswith(("s3", "gs", "hdfs", "dbfs")):
    raise FileNotFoundError(f"Raw CSV not found at {DATA_PATH}. Place it there or set APPSTORE_CSV.")

raw = (spark.read
       .option("header", "true")
       .option("multiLine", "true")      # app names/descriptions can contain line breaks
       .option("escape", '"')
       .option("mode", "PERMISSIVE")
       .csv(DATA_PATH))
raw = raw.toDF(*[c.strip().lower() for c in raw.columns])

selected, missing = [], []
for target, candidates in COLUMN_CANDIDATES.items():
    source = next((c for c in candidates if c in raw.columns), None)
    if source is None:
        missing.append(target); continue
    expr = F.col(source)
    if target == "size_mb" and source == "size_bytes":
        expr = expr.cast("double") / (1024 * 1024)
    dtype = next(f.dataType for f in CANONICAL_SCHEMA if f.name == target)
    selected.append(expr.cast(dtype).alias(target))
    print(f"  {target:<14} <- {source}")

if missing:
    raise ValueError(f"Columns not found in CSV: {missing}. Available: {raw.columns}")

df_raw = raw.select(*selected)
total_rows = df_raw.count()
print(f"\nLoaded {total_rows:,} rows x {len(df_raw.columns)} columns")
df_raw.printSchema()
df_raw.show(5, truncate=40)

## 3. Data-quality audit

### 3.1 Missing values (single pass over the data)

In [ ]:
import pandas as pd

missing_exprs = []
for f in df_raw.schema.fields:
    c = F.col(f.name)
    cond = c.isNull() | F.isnan(c) if isinstance(f.dataType, DoubleType) else \
           c.isNull() | (F.trim(c) == "") if isinstance(f.dataType, StringType) else c.isNull()
    missing_exprs.append(F.sum(F.when(cond, 1).otherwise(0)).alias(f.name))

missing_row = df_raw.agg(*missing_exprs).collect()[0].asDict()
missing_df = (pd.DataFrame({"missing": missing_row})
              .assign(pct=lambda d: (100 * d["missing"] / total_rows).round(2))
              .sort_values("missing", ascending=False))
missing_df

### 3.2 Outliers with the IQR rule (`percentile_approx`)

In [ ]:
NUMERIC_COLS = ["price", "user_rating", "rating_count", "size_mb"]
quantiles = df_raw.select(*[F.percentile_approx(c, [0.25, 0.5, 0.75], 10000).alias(c) for c in NUMERIC_COLS]).collect()[0]

rows = []
for c in NUMERIC_COLS:
    q1, q2, q3 = quantiles[c]
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = df_raw.filter((F.col(c) < lo) | (F.col(c) > hi)).count()
    rows.append({"column": c, "Q1": q1, "median": q2, "Q3": q3, "IQR": iqr,
                 "lower": lo, "upper": hi, "outliers": n_out, "pct": round(100 * n_out / total_rows, 2)})
outlier_df = pd.DataFrame(rows).set_index("column")
outlier_df

> **Interpretation:** heavy right tails in `rating_count` and `price` are genuine market behaviour (a few blockbuster apps, a few premium apps), so they are **kept** and handled later with log-scaling and banding rather than removed.

### 3.3 Business-rule validation

In [ ]:
rules = {
    "negative price":            F.col("price") < 0,
    "rating outside 0-5":        (F.col("user_rating") < 0) | (F.col("user_rating") > 5),
    "negative rating_count":     F.col("rating_count") < 0,
    "non-positive size_mb":      F.col("size_mb") <= 0,
    "empty app_name":            F.col("app_name").isNull() | (F.trim(F.col("app_name")) == ""),
}
violations = df_raw.agg(*[F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in rules.items()]).collect()[0].asDict()
duplicate_ids = df_raw.groupBy("app_id").count().filter("count > 1").count()
violations["duplicate app_id"] = duplicate_ids
pd.Series(violations, name="violations").to_frame()

## 4. Cleaning & feature engineering
Decisions:
- drop rows with an empty name or invalid rating; de-duplicate on `app_id`;
- `price` and `rating_count` nulls → 0; **unrated apps keep a null `user_rating`** (filling with 0 would bias averages down) and are flagged with `is_rated`;
- add business features used throughout the analysis.

In [ ]:
df_processed = (df_raw
    .filter(F.col("app_name").isNotNull() & (F.trim(F.col("app_name")) != ""))
    .filter(F.col("user_rating").isNull() | F.col("user_rating").between(0, 5))
    .dropDuplicates(["app_id"])
    .fillna({"price": 0.0, "rating_count": 0})
    .withColumn("user_rating", F.when(F.col("rating_count") == 0, None).otherwise(F.col("user_rating")))
    .withColumn("is_rated", F.col("user_rating").isNotNull())
    .withColumn("is_free", F.col("price") == 0.0)
    .withColumn("price_category",
        F.when(F.col("price") == 0.0, "Free")
         .when(F.col("price") <= 4.99, "Low (<$5)")
         .when(F.col("price") <= 19.99, "Medium ($5-$20)")
         .otherwise("High (>$20)"))
    .withColumn("rating_category",
        F.when(F.col("user_rating") >= 4.5, "Excellent (4.5+)")
         .when(F.col("user_rating") >= 4.0, "Good (4.0-4.4)")
         .when(F.col("user_rating") >= 3.0, "Average (3.0-3.9)")
         .when(F.col("user_rating").isNotNull(), "Poor (<3.0)")
         .otherwise("Unrated"))
    .withColumn("size_category",
        F.when(F.col("size_mb") <= 50, "Small (<=50MB)")
         .when(F.col("size_mb") <= 150, "Medium (51-150MB)")
         .otherwise("Large (>150MB)"))
    .withColumn("popularity_score", F.coalesce(F.col("rating_count") * F.col("user_rating"), F.lit(0.0)) / 1e6)
)

processed_rows = df_processed.count()
print(f"Rows before cleaning: {total_rows:,} | after: {processed_rows:,} | removed: {total_rows - processed_rows:,}")
df_processed.show(5, truncate=30)

## 5. Persist as Parquet (columnar, compressed, schema-preserving)

In [ ]:
(df_processed
    .repartition("primary_genre")          # co-locate each genre for later group-bys / windows
    .write.mode("overwrite")
    .parquet(PROCESSED_PATH))
print(f"Saved processed table -> {PROCESSED_PATH}")